# Research Paper Ingestion & Processing

In [ ]:
%pip install arxiv langchain langchain_community pypdf

In [15]:
import arxiv
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
import tempfile
import time

## Fetching Research Papers from ArXiv
- Query ArXiv for academic papers based on a search term.
- Instead of storing PDFs, we load them **temporarily** then extract the text.

In [18]:
def fetch_papers(query="LLM", max_results=5):
    """
    Fetches academic papers from ArXiv based on a search term, processes the PDFs,
    extracts text, and splits the content into smaller, structured chunks.

    Parameters:
        query (str): The search term to query ArXiv for research papers.
        max_results (int): The maximum number of papers to retrieve.

    Returns:
        all_chunks: A list of document chunks extracted from the fetched papers.
    """
    
    # Initialize an ArXiv client
    client = arxiv.Client()

    # Search query with sorting (descending order to get the latest papers)
    search = arxiv.Search(query=query, max_results=max_results, sort_order=arxiv.SortOrder.Descending)
    
    all_chunks = []  # List to store extracted text chunks
    
    for result in client.results(search):
        # Display the title of the paper currently being processed
        print(f"Processing: {result.title}")
        
        # Create a temporary file to store the downloaded PDF
        with tempfile.NamedTemporaryFile(delete=True, suffix=".pdf") as temp_pdf:
            try:
                result.download_pdf(filename=temp_pdf.name)
                print(f"-> Downloaded: {result.title}")
                
                # Load PDF file 
                loader = PyPDFLoader(temp_pdf.name)
                papers = loader.load()
                
                # Combine all pages into a single string
                full_text = " ".join(p.page_content for p in papers)
                
                # Split the extracted text into chunks
                text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
                paper_chunks = text_splitter.create_documents([full_text])
                
                # Add extracted chunks to the list
                all_chunks.extend(paper_chunks) 
                
            except (FileNotFoundError, ConnectionResetError) as e:
                print("Error occurred while downloading or processing PDF:", e)
                time.sleep(5)
    
    # Print total number of extracted chunks
    print(f"Total chunks extracted: {len(all_chunks)}")

    return all_chunks

## Running the Paper Retrieval Pipeline
Testing the function by retrieving **3 recent papers** related to **Large Language Models**.

In [ ]:
query = "Large Language Models"
paper_chunks = fetch_papers(query=query, max_results=3)

for i, chunk in enumerate(paper_chunks[:3]):
    print(f"\nChunk {i+1}:\n{chunk.page_content[:300]}...")